In [ ]:
import getpass
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

clave_pg = getpass.getpass("Contraseña de PostgreSQL: ")
engine = create_engine(URL.create(
    "postgresql+psycopg2", username="postgres", password=clave_pg,
    host="localhost", port=5432, database="tesis_agronomia"))

tablas = ["estudiantes", "actas_cursada", "examenes",
          "insc_cursada", "insc_examen", "equivalencias"]
datos = {t: pd.read_sql(f"SELECT * FROM tesis.{t}", engine) for t in tablas}

for t, df in datos.items():
    print(f"{t}: {len(df)} filas")

In [ ]:
CORTE = 21  # 2.º cuatrimestre de 2025

def num_cuatri(anio, c):
    return (anio - 2015) * 2 + (c - 1)

def por_fecha(fecha):
    f = pd.Timestamp(fecha)
    if 4 <= f.month <= 8:
        return num_cuatri(f.year, 1)
    if f.month >= 9:
        return num_cuatri(f.year, 2)
    return num_cuatri(f.year - 1, 2)          # enero a marzo: año anterior

def cuatris_de_cursada(fila, col_fecha):
    p = fila["periodo"].lower()
    a = fila["anio_academico"]
    if "variable" in p:
        return [por_fecha(fila[col_fecha])]
    if "anual" in p:
        return [num_cuatri(a, 1), num_cuatri(a, 2)]
    if "bimestre" in p or p.startswith("1"):
        return [num_cuatri(a, 1)]
    if p.startswith("2"):
        return [num_cuatri(a, 2)]
    return [None]                             # período no reconocido

filas = []
for nombre, df in [("actas_cursada", datos["actas_cursada"]),
                   ("insc_cursada", datos["insc_cursada"])]:
    for _, r in df.iterrows():
        for q in cuatris_de_cursada(r, "fecha"):
            filas.append((r["id_est"], q, nombre))
for nombre in ["examenes", "insc_examen"]:
    for _, r in datos[nombre].iterrows():
        filas.append((r["id_est"], por_fecha(r["fecha"]), nombre))

act = pd.DataFrame(filas, columns=["id_est", "cuatri", "fuente"])

print("Períodos no reconocidos:", act["cuatri"].isna().sum())
act = act.dropna(subset=["cuatri"]).astype({"cuatri": int})

est = datos["estudiantes"]
act = act.merge(est[["id_est", "cohorte"]], on="id_est")
act["inicio"] = act["cohorte"].apply(lambda a: num_cuatri(a, 1))
print("Registros antes del ingreso:", (act["cuatri"] < act["inicio"]).sum())
print("Registros después del corte:", (act["cuatri"] > CORTE).sum())

act = act[(act["cuatri"] >= act["inicio"]) & (act["cuatri"] <= CORTE)]
actividad = act[["id_est", "cuatri"]].drop_duplicates()
print("\nCuatrimestres con actividad (estudiante-cuatrimestre):", len(actividad))
print("Estudiantes con actividad:", actividad["id_est"].nunique())

In [ ]:
# --- Detalle: registro anterior al ingreso y registros posteriores al corte ---
todo = pd.DataFrame(filas, columns=["id_est", "cuatri", "fuente"]).dropna(subset=["cuatri"])
todo = todo.astype({"cuatri": int}).merge(est[["id_est", "cohorte"]], on="id_est")
todo["inicio"] = todo["cohorte"].apply(lambda a: num_cuatri(a, 1))

print("Registro anterior al ingreso:")
print(todo[todo["cuatri"] < todo["inicio"]].to_string(index=False), "\n")

print("Registros posteriores al corte, por fuente:")
print(todo[todo["cuatri"] > CORTE]["fuente"].value_counts().to_string())

In [ ]:
pausas = []
for id_est, g in actividad.sort_values("cuatri").groupby("id_est"):
    c = g["cuatri"].tolist()
    for a, b in zip(c, c[1:]):
        if b - a > 1:
            pausas.append(b - a - 1)

pausas = pd.Series(pausas, name="duracion")
print("Pausas con regreso posterior:", len(pausas))
print("\nDuración (cuatrimestres sin actividad) -> cantidad de pausas")
tabla = pausas.clip(upper=6).value_counts().sort_index()
tabla.index = [str(i) if i < 6 else "6 o más" for i in tabla.index]
print(tabla.to_string())
print("\nPausas que se confundirían con deserción según N (duran N o más):")
for n in [2, 3, 4, 5, 6]:
    print(f"  N = {n}: {(pausas >= n).mean()*100:.1f} %  ({(pausas >= n).sum()} pausas)")

In [ ]:
ultima = actividad.groupby("id_est")["cuatri"].max().rename("ultima")
base = est.set_index("id_est").join(ultima)
egresado = base["fecha_egreso"].notna()

print("N | desertores | censurados | egresados")
for n in [2, 3, 4, 5]:
    deserta = (~egresado) & (CORTE - base["ultima"] >= n)
    censurado = (~egresado) & ~deserta
    print(f"{n} | {deserta.sum():10d} | {censurado.sum():10d} | {egresado.sum():9d}")

In [ ]:
N = 3
base["inicio"] = base["cohorte"].apply(lambda a: num_cuatri(a, 1))
base["deserta"] = (~egresado) & (CORTE - base["ultima"] >= N)
base["deserta_1er_anio"] = base["deserta"] & (base["ultima"] <= base["inicio"] + 1)
base["grupo"] = "modelo"
base.loc[base["deserta_1er_anio"], "grupo"] = "desertó en 1.er año"

resumen = base.groupby(["cohorte", "grupo"]).size().unstack(fill_value=0)
resumen["total"] = resumen.sum(axis=1)
resumen.loc["Total"] = resumen.sum()
print(resumen.to_string())

In [ ]:
mod = base[base["grupo"] == "modelo"].copy()

mod["cuatri_egreso"] = mod["fecha_egreso"].apply(
    lambda f: por_fecha(f) if pd.notna(f) else None)
mod["fin"] = mod["cuatri_egreso"].fillna(mod["ultima"]).astype(int)
mod["tiempo"] = mod["fin"] - (mod["inicio"] + 1)
mod["evento"] = mod["deserta"].astype(int)

print("Desertores (evento = 1):", mod["evento"].sum())
print("Censurados (evento = 0):", (mod["evento"] == 0).sum())
print("\nTiempo (cuatrimestres desde el final del 1.er año) -> estudiantes")
print(mod.groupby(["tiempo", "evento"]).size().unstack(fill_value=0).to_string())

In [ ]:
trayectorias = base.reset_index()[[
    "id_est", "cohorte", "sede_ingreso", "inicio", "ultima",
    "deserta", "deserta_1er_anio", "grupo"]].copy()
trayectorias = trayectorias.merge(
    mod.reset_index()[["id_est", "tiempo", "evento"]], on="id_est", how="left")

trayectorias.to_sql("trayectorias", engine, schema="tesis",
                    if_exists="replace", index=False)

print(pd.read_sql("SELECT grupo, COUNT(*) AS estudiantes, "
                  "SUM(evento) AS desertores FROM tesis.trayectorias "
                  "GROUP BY grupo", engine).to_string(index=False))

In [ ]:
ac = datos["actas_cursada"].copy()

def periodo_unificado(p):
    p = p.lower()
    if "variable" in p: return "Variable"
    if "anual" in p:    return "Anual"
    if "bimestre" in p: return "1er bimestre"
    if p.startswith("1"): return "1er cuatrimestre"
    if p.startswith("2"): return "2do cuatrimestre"
ac["periodo"] = ac["periodo"].apply(periodo_unificado)

clave = ["id_est", "materia", "anio_academico", "periodo"]

# cursadas con más de un acta de regularidad (para controlar)
n_reg = ac[ac["origen"] == "R"].groupby(clave).size()
print("Cursadas con más de un acta de regularidad:", (n_reg > 1).sum())

# acta de regularidad: si hay más de una, se toma la más reciente
reg = (ac[ac["origen"] == "R"].sort_values("fecha")
         .groupby(clave).tail(1)
         .set_index(clave)[["resultado", "fecha"]])
prom = (ac[ac["origen"] == "P"].sort_values("fecha")
          .groupby(clave).tail(1)
          .set_index(clave)[["nota", "fecha"]])

cursadas = (reg.rename(columns={"fecha": "fecha_reg"})
              .join(prom.rename(columns={"fecha": "fecha_prom"}), how="outer"))
nombres = {"A": "regular", "R": "libre", "U": "ausente"}
cursadas["estado"] = cursadas["resultado"].map(nombres)
cursadas.loc[cursadas["nota"].notna(), "estado"] = "promocionada"
cursadas["nota_promocion"] = pd.to_numeric(cursadas["nota"], errors="coerce")
cursadas["fecha"] = cursadas["fecha_prom"].fillna(cursadas["fecha_reg"])
cursadas = cursadas.reset_index()[clave + ["estado", "nota_promocion", "fecha"]]

print("Total de cursadas:", len(cursadas))
print(cursadas["estado"].value_counts().to_string())

In [ ]:
ex = datos["examenes"]

promos = cursadas[cursadas["estado"] == "promocionada"][
    ["id_est", "materia", "fecha", "nota_promocion"]].rename(
    columns={"nota_promocion": "nota"})
promos["via"] = "promoción"

finales = ex[ex["resultado"] == "A"][["id_est", "materia", "fecha", "nota"]].copy()
finales["nota"] = pd.to_numeric(finales["nota"], errors="coerce")
finales["via"] = "examen"

aprob = pd.concat([promos, finales]).sort_values("fecha")
dup = aprob.duplicated(["id_est", "materia"], keep="first")
print("Materias aprobadas más de una vez (se conserva la primera):", dup.sum())
aprobaciones = aprob[~dup]

print("Total de aprobaciones:", len(aprobaciones))
print(aprobaciones["via"].value_counts().to_string())
print("Aprobaciones sin nota numérica:", aprobaciones["nota"].isna().sum())

In [ ]:
sin_nota = aprobaciones[aprobaciones["nota"].isna()]
print("Aprobaciones sin nota numérica, por vía:")
print(sin_nota["via"].value_counts().to_string())

actas_p = ac[ac["origen"] == "P"].copy()
actas_p["nota_num"] = pd.to_numeric(actas_p["nota"], errors="coerce")
no_num = actas_p[actas_p["nota_num"].isna()]

print("\nActas de promoción con nota no numérica:", len(no_num))
print("Valor original de la nota:")
print(no_num["nota"].value_counts(dropna=False).to_string())

print("\nPor año académico:")
print(no_num["anio_academico"].value_counts().sort_index().to_string())

In [ ]:
mat = pd.read_sql("SELECT DISTINCT materia, nombre FROM tesis.materias", engine)
mat = mat.drop_duplicates("materia")

conteo = (no_num.groupby("materia").size().rename("promociones_con_A")
          .reset_index().merge(mat, on="materia", how="left")
          .sort_values("promociones_con_A", ascending=False))
print("Materias distintas:", len(conteo))
print(conteo.head(15).to_string(index=False))

In [ ]:
cursadas["cuatri"] = cursadas.apply(
    lambda r: min(cuatris_de_cursada(r, "fecha")), axis=1)
aprobaciones = aprobaciones.copy()
aprobaciones["cuatri"] = aprobaciones["fecha"].apply(por_fecha)

cursadas.to_sql("cursadas", engine, schema="tesis", if_exists="replace", index=False)
aprobaciones.to_sql("aprobaciones", engine, schema="tesis", if_exists="replace", index=False)

for t in ["cursadas", "aprobaciones"]:
    n = pd.read_sql(f"SELECT COUNT(*) AS filas FROM tesis.{t}", engine)["filas"][0]
    print(f"tesis.{t}: {n} filas")